# Neural Network with Gradient Descent and Backpropagation

This notebook implements a simple neural network from scratch with:
- Different activation functions (sigmoid, tanh, ReLU)
- Gradient descent optimization
- Backpropagation algorithm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

## Activation Functions and Their Derivatives

In [ ]:
class ActivationFunctions:
    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    @staticmethod
    def sigmoid_derivative(x):
        s = ActivationFunctions.sigmoid(x)
        return s * (1 - s)
    
    @staticmethod
    def tanh(x):
        return np.tanh(x)
    
    @staticmethod
    def tanh_derivative(x):
        return 1 - np.tanh(x) ** 2
    
    @staticmethod
    def relu(x):
        return np.maximum(0, x)
    
    @staticmethod
    def relu_derivative(x):
        return (x > 0).astype(float)

## Neural Network Class

In [ ]:
class SimpleNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, activation='sigmoid', learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        
        # Initialize weights randomly
        self.W1 = np.random.randn(self.input_size, self.hidden_size) * 0.5
        self.b1 = np.zeros((1, self.hidden_size))
        self.W2 = np.random.randn(self.hidden_size, self.output_size) * 0.5
        self.b2 = np.zeros((1, self.output_size))
        
        # Set activation function
        self.activation_name = activation
        if activation == 'sigmoid':
            self.activation = ActivationFunctions.sigmoid
            self.activation_derivative = ActivationFunctions.sigmoid_derivative
        elif activation == 'tanh':
            self.activation = ActivationFunctions.tanh
            self.activation_derivative = ActivationFunctions.tanh_derivative
        elif activation == 'relu':
            self.activation = ActivationFunctions.relu
            self.activation_derivative = ActivationFunctions.relu_derivative
        
        self.losses = []
    
    def forward(self, X):
        # Forward propagation
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.activation(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = ActivationFunctions.sigmoid(self.z2)  # Output layer always sigmoid for binary classification
        return self.a2
    
    def backward(self, X, y, output):
        # Backpropagation
        m = X.shape[0]
        
        # Output layer gradients
        dz2 = output - y
        dW2 = (1/m) * np.dot(self.a1.T, dz2)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)
        
        # Hidden layer gradients
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * self.activation_derivative(self.z1)
        dW1 = (1/m) * np.dot(X.T, dz1)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)
        
        # Update weights using gradient descent
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
    
    def compute_loss(self, y_true, y_pred):
        # Binary cross-entropy loss
        m = y_true.shape[0]
        loss = -(1/m) * np.sum(y_true * np.log(y_pred + 1e-8) + (1 - y_true) * np.log(1 - y_pred + 1e-8))
        return loss
    
    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            # Forward pass
            output = self.forward(X)
            
            # Compute loss
            loss = self.compute_loss(y, output)
            self.losses.append(loss)
            
            # Backward pass
            self.backward(X, y, output)
            
            if epoch % 100 == 0:
                print(f'Epoch {epoch}, Loss: {loss:.4f}')
    
    def predict(self, X):
        output = self.forward(X)
        return (output > 0.5).astype(int)
    
    def accuracy(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == y)

## Generate Sample Data and Train Networks

In [ ]:
# Generate sample data
X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0, n_informative=2, 
                          n_clusters_per_class=1, random_state=42)
y = y.reshape(-1, 1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")

In [ ]:
# Train networks with different activation functions
activations = ['sigmoid', 'tanh', 'relu']
networks = {}

for activation in activations:
    print(f"\nTraining network with {activation} activation:")
    print("-" * 50)
    
    # Create and train network
    nn = SimpleNeuralNetwork(input_size=2, hidden_size=10, output_size=1, 
                           activation=activation, learning_rate=0.1)
    nn.train(X_train, y_train, epochs=1000)
    
    # Evaluate
    train_acc = nn.accuracy(X_train, y_train)
    test_acc = nn.accuracy(X_test, y_test)
    
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    
    networks[activation] = nn

## Visualize Results

In [ ]:
# Plot loss curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for activation in activations:
    plt.plot(networks[activation].losses, label=f'{activation}')
plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot decision boundaries
plt.subplot(1, 2, 2)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test.ravel(), cmap='viridis', alpha=0.7)
plt.title('Test Data Distribution')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar()

plt.tight_layout()
plt.show()

## Compare Activation Functions

In [ ]:
# Compare final performance
print("\nFinal Performance Comparison:")
print("=" * 50)

for activation in activations:
    nn = networks[activation]
    train_acc = nn.accuracy(X_train, y_train)
    test_acc = nn.accuracy(X_test, y_test)
    final_loss = nn.losses[-1]
    
    print(f"{activation.upper():>8}: Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}, Final Loss: {final_loss:.4f}")

## Visualize Activation Functions

In [ ]:
# Plot activation functions
x = np.linspace(-5, 5, 100)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(x, ActivationFunctions.sigmoid(x), 'b-', label='Sigmoid')
plt.plot(x, ActivationFunctions.sigmoid_derivative(x), 'r--', label='Sigmoid Derivative')
plt.title('Sigmoid Function')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(x, ActivationFunctions.tanh(x), 'b-', label='Tanh')
plt.plot(x, ActivationFunctions.tanh_derivative(x), 'r--', label='Tanh Derivative')
plt.title('Tanh Function')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(x, ActivationFunctions.relu(x), 'b-', label='ReLU')
plt.plot(x, ActivationFunctions.relu_derivative(x), 'r--', label='ReLU Derivative')
plt.title('ReLU Function')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()